In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import medfilt
from scipy.ndimage import maximum_filter1d

# Paramètres
sampling_rate = 128  # Hz
min_peak_value = 10  # °/s
merge_window = int(1 * sampling_rate)  # 1 seconde
gap_threshold = int(0.5 * sampling_rate)  # 0.5 seconde
min_duration = int(1.5 * sampling_rate)  # 1.5 seconde

# 1. Extraire les 3 axes gyroscopiques
gyro_x = X_clean[:, 0]
gyro_y = X_clean[:, 1]
gyro_z = X_clean[:, 2]

# 2. Détection des pics > 10°/s sur chaque axe
def get_axis_peaks(axis_data):
    return np.abs(axis_data[np.abs(axis_data) > min_peak_value])

peaks_x = get_axis_peaks(gyro_x)
peaks_y = get_axis_peaks(gyro_y)
peaks_z = get_axis_peaks(gyro_z)

# 3. Moyenne des pics par axe
mean_x = peaks_x.mean() if len(peaks_x) > 0 else np.inf
mean_y = peaks_y.mean() if len(peaks_y) > 0 else np.inf
mean_z = peaks_z.mean() if len(peaks_z) > 0 else np.inf

# 4. Seuil adaptatif = min des moyennes
adaptive_threshold = min(mean_x, mean_y, mean_z)
print(f"Seuil adaptatif : {adaptive_threshold:.2f} °/s")

# 5. Détection brute : un mouvement si un axe dépasse le seuil
movement_raw = (
    (np.abs(gyro_x) > adaptive_threshold) |
    (np.abs(gyro_y) > adaptive_threshold) |
    (np.abs(gyro_z) > adaptive_threshold)
).astype(int)

# 6. Filtre max mobile (fusionner les mouvements séparés de < 0.5s)
movement_merged = maximum_filter1d(movement_raw, size=merge_window)

# 7. Filtre médian mobile (supprimer les mouvements < 1.5s)
movement_final = medfilt(movement_merged, kernel_size=min_duration | 1)  # kernel must be odd

# 8. Visualisation
plt.figure(figsize=(12, 4))
plt.plot(np.linalg.norm(X_clean[:, :3], axis=1), label="Norme Gyro")
plt.plot(movement_final * adaptive_threshold, label="Mouvement détecté", color='orange')
plt.axhline(adaptive_threshold, color='r', linestyle='--', label="Seuil adaptatif")
plt.legend()
plt.title("Détection de mouvement du bras (seuil adaptatif)")
plt.xlabel("Échantillons")
plt.ylabel("Vitesse angulaire (°/s)")
plt.tight_layout()
plt.show()
